In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import joblib


In [ ]:
df = pd.read_csv("../data.csv")
df.head()


In [ ]:

df = df[
    ["price", "city", "location", "bedrooms", "baths", "Area Type", "Area Size"]
].copy()



In [ ]:

df.rename(columns={
    "price": "Price",
    "city": "City",
    "location": "Location",
    "bedrooms": "Bedrooms",
    "baths": "Bathrooms",
    "Area Size": "AreaSize",
    "Area Type": "AreaType"
}, inplace=True)

df.head()



In [ ]:
df["Area_sqft"] = df.apply(
    lambda row: row["AreaSize"] * 272.25 if row["AreaType"] == "Marla" else row["AreaSize"] * 5445,
    axis=1
)


In [ ]:

df = df[["Area_sqft", "Bedrooms", "Bathrooms", "City", "Location", "Price"]]
df.dropna(inplace=True)
df.head()



In [ ]:
plt.hist(df["Price"], bins=30)
plt.title("House Price Distribution")
plt.show()

plt.scatter(df["Area_sqft"], df["Price"])
plt.xlabel("Area (sq ft)")
plt.ylabel("Price")
plt.show()


In [ ]:
X = df.drop("Price", axis=1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"),
         ["Area_sqft", "Bedrooms", "Bathrooms"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["City", "Location"])
    ]
)



In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)


In [ ]:
predictions = model.predict(X_test)
print("R² Score:", r2_score(y_test, predictions))


In [ ]:
joblib.dump(model, "../house_model.pkl")
print("Model saved successfully")
